In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers, Model, callbacks
import ruptures as rpt
import matplotlib.pyplot as plt
from pyod.models.iforest import IForest
from pyod.models.lof import LOF
from pyod.models.ecod import ECOD
from pyod.models.hbos import HBOS
from pyod.models.knn import KNN
from pyod.models.kde import KDE
from scipy.stats import kurtosis, skew, chi2, entropy, t, lognorm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.datasets import make_blobs
from sklearn.svm import OneClassSVM
import time
import warnings
from sklearn.exceptions import UndefinedMetricWarning

# --- Suppress Warnings ---
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
tf.get_logger().setLevel('ERROR')

# ---
# --- MODEL: T2Detector (User Provided) ---
# ---
class T2Detector:
    def __init__(self, alpha=0.05):
        self.alpha = alpha
        self.mean_ = None
        self.cov_ = None
        self.inv_cov_ = None
        self.threshold_ = None

    def fit(self, X, y=None):
        self.mean_ = np.mean(X, axis=0)
        self.cov_ = np.cov(X, rowvar=False)
        if np.ndim(self.cov_) == 0:
            self.cov_ = np.array([[self.cov_]])
        self.inv_cov_ = np.linalg.pinv(self.cov_)
        df = X.shape[1]
        self.threshold_ = chi2.ppf(1 - self.alpha, df=df)
        return self

    def predict(self, X):
        if X.ndim == 1:
            X = X.reshape(1, -1)
        if self.mean_.ndim == 0:
            self.mean_ = self.mean_.reshape(1)
            
        diff = X - self.mean_
        
        if diff.ndim == 1:
            diff = diff.reshape(1, -1)
            
        if self.inv_cov_.ndim == 0:
            self.inv_cov_ = self.inv_cov_.reshape(1, 1)
        elif self.inv_cov_.ndim == 1:
            self.inv_cov_ = self.inv_cov_.reshape(1, -1)
            
        if X.shape[1] == 1 and self.inv_cov_.shape == ():
             self.inv_cov_ = np.array([[self.inv_cov_]])
        
        md2 = np.sum(diff @ self.inv_cov_ * diff, axis=1)
        return (md2 > self.threshold_).astype(int)

# ---
# --- MODEL: MEWMA_Detector (NEW) ---
# ---
class MEWMA_Detector:
    """
    Multivariate Exponentially Weighted Moving Average (MEWMA) Detector.
    
    This detector is stateful and tracks the process mean over time.
    It is effective at detecting small, sustained shifts.
    """
    def __init__(self, alpha=0.005, lambda_val=0.1):
        self.alpha = alpha
        self.lambda_ = lambda_val # Smoothing parameter
        self.mean_ = None
        self.cov_ = None
        self.inv_cov_ = None
        self.p_ = None # num features

    def fit(self, X, y=None):
        self.p_ = X.shape[1]
        self.mean_ = np.mean(X, axis=0)
        self.cov_ = np.cov(X, rowvar=False)
        if np.ndim(self.cov_) == 0:
            self.cov_ = np.array([[self.cov_]])
        self.inv_cov_ = np.linalg.pinv(self.cov_)
        return self

    def predict(self, X):
        n = X.shape[0]
        preds = np.zeros(n, dtype=int)
        
        # Initialize z_i (EWMA vector) to the Phase I mean
        z_i = self.mean_ 
        
        # Asymptotic control limit (standard practice)
        UCL = chi2.ppf(1 - self.alpha, df=self.p_)
        
        for i in range(n):
            x_i = X[i, :]
            
            # Update EWMA vector
            z_i = (1 - self.lambda_) * z_i + self.lambda_ * x_i
            
            # Calculate the exact variance factor for T2 at time i
            # This accounts for the "start-up" effect
            C_i_factor = (self.lambda_ / (2 - self.lambda_)) * (1 - (1 - self.lambda_)**(2 * (i + 1)))
            
            if np.isclose(C_i_factor, 0):
                 T2_i = 0.0
            else:
                 # T2_i = (z_i - mu).T @ [Cov(z_i)]^-1 @ (z_i - mu)
                 # T2_i = (z_i - mu).T @ [C_i * Cov(X)]^-1 @ (z_i - mu)
                 # T2_i = (1 / C_i) * (z_i - mu).T @ [Cov(X)]^-1 @ (z_i - mu)
                 diff = z_i - self.mean_
                 T2_i = (1 / C_i_factor) * (diff.T @ self.inv_cov_ @ diff)
            
            if T2_i > UCL:
                preds[i] = 1
                
        return preds

# ---
# --- MODEL: OC_SVM_Detector (NEW) ---
# ---
class OC_SVM_Detector:
    """
    One-Class SVM Detector for novelty detection.
    """
    def __init__(self, alpha=0.005):
        # 'nu' is an upper bound on the fraction of training errors (anomalies in train set)
        # and a lower bound of the fraction of support vectors.
        # Since Phase I is "clean", we set nu to our expected FPR (alpha)
        self.nu = alpha
        self.model = OneClassSVM(gamma='auto', nu=self.nu) 

    def fit(self, X, y=None):
        self.model.fit(X)
        return self

    def predict(self, X):
        # model.predict returns 1 for inliers, -1 for outliers
        preds = self.model.predict(X)
        return (preds == -1).astype(int) # Convert to 1=outlier, 0=inlier


In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import IsolationForest

# ----------------------------------------------------------
# Load SECOM Data
# ----------------------------------------------------------
# Expecting files:
#   secom.data    -> features
#   secom_labels.data -> labels (1 = fail, -1 = pass)
# ----------------------------------------------------------

data = pd.read_csv("uci-secom.csv")
X = data.drop(['Time', 'Pass/Fail'], axis = 1)
y = data['Pass/Fail']

y = y.values
# Convert SECOM labels:
# 1 = abnormal/fail (POSITIVE)
# -1 = normal (NEGATIVE)
y = np.where(y == 1, 1, 0)

# ----------------------------------------------------------
# Preprocessing: Handle NaNs + scale
# ----------------------------------------------------------
imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X)

scaler = StandardScaler()
X= scaler.fit_transform(X_imputed)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks
from scipy.stats import chi2

class VAE:
    def __init__(
        self, 
        latent_dim=2, 
        encoder_layers=(64, 32), 
        decoder_layers=(32, 64),
        learning_rate=1e-3, 
        alpha=0.05, 
        combine_rule="or", 
        use_sidak=True
    ):
        self.latent_dim = latent_dim
        self.encoder_layers = encoder_layers
        self.decoder_layers = decoder_layers
        self.learning_rate = learning_rate
        self.alpha = alpha
        self.combine_rule = combine_rule  # "or" or "and"
        self.use_sidak = use_sidak

        # Will be created later
        self.vae = None
        self.encoder = None
        self.decoder = None

        # Statistical parameters
        self.latent_mean = None
        self.latent_cov_inv = None
        self.T2_threshold = None
        self.SPE_threshold = None

    # ----------------------------------------------------
    # Build encoder and decoder
    # ----------------------------------------------------
    def _build(self, input_dim):
        inputs = layers.Input(shape=(input_dim,))
        x = inputs

        for units in self.encoder_layers:
            x = layers.Dense(units, activation="relu")(x)

        z_mean = layers.Dense(self.latent_dim, name="z_mean")(x)
        z_log_var = layers.Dense(self.latent_dim, name="z_log_var")(x)

        def sampling(args):
            zm, zv = args
            eps = tf.random.normal(shape=tf.shape(zm))
            return zm + tf.exp(0.5 * zv) * eps

        z = layers.Lambda(sampling, name="z")([z_mean, z_log_var])

        self.encoder = Model(inputs, [z_mean, z_log_var, z], name="encoder")

        latent_inputs = layers.Input(shape=(self.latent_dim,))
        y = latent_inputs
        for units in self.decoder_layers:
            y = layers.Dense(units, activation="relu")(y)
        outputs = layers.Dense(input_dim, activation="linear")(y)
        self.decoder = Model(latent_inputs, outputs, name="decoder")

        class VAECore(Model):
            def __init__(self, encoder, decoder):
                super().__init__()
                self.encoder = encoder
                self.decoder = decoder
            def call(self, inputs):
                z_mean, z_log_var, z = self.encoder(inputs)
                return self.decoder(z)

        self.vae = VAECore(self.encoder, self.decoder)
        self.vae.compile(
            optimizer=tf.keras.optimizers.Adam(self.learning_rate),
            loss='mse'
        )

    # ----------------------------------------------------
    # TRAIN MODEL
    # ----------------------------------------------------
    def fit(self, X, epochs=50, batch_size=32, verbose=0):
        input_dim = X.shape[1]
        self._build(input_dim)

        early_stop = callbacks.EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
        self.vae.fit(X, X, epochs=epochs, batch_size=batch_size, verbose=verbose, callbacks=[early_stop])

        # Latent outputs
        z_mean, _, _ = self.encoder.predict(X, verbose=0)
        X_recon = self.decoder.predict(z_mean, verbose=0)

        # Compute T² stats
        self.latent_mean = np.mean(z_mean, axis=0)
        cov = np.cov(z_mean, rowvar=False)
        self.latent_cov_inv = np.linalg.pinv(cov)

        T2_scores = np.einsum("ij,jk,ik->i", z_mean - self.latent_mean, self.latent_cov_inv, z_mean - self.latent_mean)

        # Compute SPE stats
        SPE_scores = np.sum((X - X_recon) ** 2, axis=1)

        # Adjust alpha using Šidák or Bonferroni
        if self.use_sidak:
            alpha_star = 1 - (1 - self.alpha)**(1/2)
        else:
            alpha_star = self.alpha / 2

        # Thresholds
        self.T2_threshold = chi2.ppf(1 - alpha_star, df=self.latent_dim)
        self.SPE_threshold = np.quantile(SPE_scores, 1 - alpha_star)

    # ----------------------------------------------------
    # OUTLIER DECISION INTERFACE
    # ----------------------------------------------------
    def is_outlier(self, X):
        z_mean, _, _ = self.encoder.predict(X, verbose=0)
        X_recon = self.decoder.predict(z_mean, verbose=0)

        # Stats
        T2_scores = np.einsum("ij,jk,ik->i", z_mean - self.latent_mean, self.latent_cov_inv, z_mean - self.latent_mean)
        SPE_scores = np.sum((X - X_recon) ** 2, axis=1)

        T2_flag = T2_scores > self.T2_threshold
        SPE_flag = SPE_scores > self.SPE_threshold

        if self.combine_rule == "and":
            outlier = np.logical_and(T2_flag, SPE_flag)
        else:
            outlier = np.logical_or(T2_flag, SPE_flag)

        return outlier, T2_scores, SPE_scores, self.T2_threshold, self.SPE_threshold

    # ----------------------------------------------------
    # PyOD-style PREDICT (required by your benchmark)
    # ----------------------------------------------------
    def predict(self, X):
        """Return 1 for outlier, 0 for inlier."""
        out, *_ = self.is_outlier(X)
        return out.astype(int)

    # ----------------------------------------------------
    # PyOD-style SCORE FUNCTION (required for AUROC)
    # ----------------------------------------------------
    def decision_function(self, X):
        """Return an anomaly score: higher = more anomalous."""

        z_mean, _, _ = self.encoder.predict(X, verbose=0)
        X_recon = self.decoder.predict(z_mean, verbose=0)

        T2_scores = np.einsum("ij,jk,ik->i", z_mean - self.latent_mean, self.latent_cov_inv, z_mean - self.latent_mean)
        SPE_scores = np.sum((X - X_recon) ** 2, axis=1)

        # Standardize both to comparable scale
        T2_norm = (T2_scores - np.min(T2_scores)) / (np.ptp(T2_scores) + 1e-8)
        SPE_norm = (SPE_scores - np.min(SPE_scores)) / (np.ptp(SPE_scores) + 1e-8)

        # Combined score
        score = np.maximum(T2_norm, SPE_norm)
        return score

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# PyOD imports
from pyod.models.iforest import IForest
from pyod.models.lof import LOF
from pyod.models.ecod import ECOD
from pyod.models.knn import KNN
from pyod.models.suod import SUOD

# ==========================================================
# METRIC COMPUTATION
# ==========================================================
def compute_metrics(y_true, is_out, scores):
    """
    is_out = 1 for anomaly, 0 for normal
    """
    recall = np.mean(is_out[y_true == 1]) if np.sum(y_true == 1) > 0 else 0.0
    precision = np.mean(y_true[is_out == 1] == 1) if np.sum(is_out == 1) > 0 else 0.0
    fpr = np.mean(is_out[y_true == 0]) if np.sum(y_true == 0) > 0 else 0.0
    inlier_ret = np.mean(is_out[y_true == 0] == 0) if np.sum(y_true == 0) > 0 else 0.0
    auroc = roc_auc_score(y_true, scores) if len(np.unique(y_true)) == 2 else 0.5
    return recall, precision, fpr, inlier_ret, auroc

# ==========================================================
# BENCHMARK FUNCTION (MODIFIED FOR SINGLE DATASET)
# ==========================================================
def run_benchmark_on_data(X, y, dataset_name, models, scaler=StandardScaler(), runs=5):
    """
    Runs models on pre-loaded X and y arrays.
    """
    
    # Scale the data
    X_scaled = scaler.fit_transform(X)
    
    rows = []

    for mdl_name, mdl_fn in models.items():
        print(f"  Running model: {mdl_name}...")

        # Store the metric history across runs
        rec_list, prec_list, fpr_list, inret_list, auc_list = [], [], [], [], []

        for r in range(runs):
            try:
                model = mdl_fn()

                # ---------------------------------------------
                # Special case: VSCOUT / VAE (Keras/Tensorflow models)
                # ---------------------------------------------
                if mdl_name.upper() in ["VSCOUT", "VAE"]:
                    # Assuming fit signature is similar for both
                    model.fit(X_scaled, epochs=110, batch_size=64, verbose=0)

                    out = model.is_outlier(X_scaled)
                    
                    # Handle tuple return (is_outlier, ...) vs single array
                    if isinstance(out, tuple):
                        is_out = np.array(out[0]).astype(int)
                    else:
                        is_out = np.array(out).astype(int)

                    # Scores
                    if hasattr(model, "decision_function"):
                        scores = model.decision_function(X_scaled)
                    else:
                        # Fallback if decision_function not implemented
                        scores = is_out.astype(float)

                # ---------------------------------------------
                # Standard PyOD models
                # ---------------------------------------------
                else:
                    model.fit(X_scaled)

                    if hasattr(model, "predict"):
                        is_out = model.predict(X_scaled).astype(int)
                    else:
                        raise ValueError(f"No usable predict() for {mdl_name}")

                    if hasattr(model, "decision_function"):
                        scores = model.decision_function(X_scaled)
                    else:
                        scores = is_out.astype(float)

                # Compute metrics
                recall, precision, fpr, inlier_ret, auroc = compute_metrics(y, is_out, scores)

                rec_list.append(recall)
                prec_list.append(precision)
                fpr_list.append(fpr)
                inret_list.append(inlier_ret)
                auc_list.append(auroc)
            
            except Exception as e:
                print(f"    Error running {mdl_name} run {r}: {e}")
                continue

        # If all runs failed, skip adding row
        if not rec_list:
            continue

        # Store means + stds
        rows.append(dict(
            Dataset=dataset_name,
            Model=mdl_name,
            Recall_mean=np.mean(rec_list), Recall_std=np.std(rec_list),
            Precision_mean=np.mean(prec_list), Precision_std=np.std(prec_list),
            FPR_mean=np.mean(fpr_list), FPR_std=np.std(fpr_list),
            InlierRetention_mean=np.mean(inret_list), InlierRetention_std=np.std(inret_list),
            AUROC_mean=np.mean(auc_list), AUROC_std=np.std(auc_list),
        ))

    return rows

# ==========================================================
# LATEX TABLE GENERATOR
# ==========================================================
def make_latex_table(results_df):
    def fmt(mean, std):
        return f"{mean:.4f} ({std:.2f})"

    L = []
    L.append("\\begin{table}[H]")
    L.append("\\centering")
    L.append("\\scriptsize")
    L.append("\\caption{Comparison on Single Dataset.}")
    L.append("\\label{tab:single-results}")
    L.append("\\resizebox{0.8\\textwidth}{!}{")
    L.append("\\begin{tabular}{lccccc}")
    L.append("\\toprule")
    L.append("Model & Recall & Precision & FPR & Inlier Retention & AUROC \\\\")
    L.append("\\midrule")

    for _, row in results_df.iterrows():
        L.append(
            f"{row['Model']} & "
            f"{fmt(row['Recall_mean'], row['Recall_std'])} & "
            f"{fmt(row['Precision_mean'], row['Precision_std'])} & "
            f"{fmt(row['FPR_mean'], row['FPR_std'])} & "
            f"{fmt(row['InlierRetention_mean'], row['InlierRetention_std'])} & "
            f"{fmt(row['AUROC_mean'], row['AUROC_std'])} \\\\"
        )

    L.append("\\bottomrule")
    L.append("\\end{tabular}")
    L.append("}")
    L.append("\\end{table}")

    return "\n".join(L)


# ==========================================================
# MAIN EXECUTION
# ==========================================================

models = {
    "VSCOUT": lambda: VSCOUT(alpha=0.05, flag_rule="any"),
    "VAE": lambda: VAE(alpha=0.05),
    "IForest": lambda: IForest(contamination=0.05),
    "LOF": lambda: LOF(contamination=0.05),
    "ECOD": lambda: ECOD(contamination=0.05),
    "KNN": lambda: KNN(contamination=0.05),
    "SUOD": lambda: SUOD(
        base_estimators=[
            IForest(contamination=0.05),
            LOF(contamination=0.05),
            ECOD(contamination=0.05),
            KNN(contamination=0.05),
        ],
        n_jobs=1,
        combination="average",
        contamination=0.05
    )
}


dataset_name = "SECOM"

# 3. RUN BENCHMARK
print(f"Starting benchmark for {dataset_name}...")
rows = run_benchmark_on_data(X, y, dataset_name, models, runs=10)

# 4. DISPLAY RESULTS
if rows:
    results_df = pd.DataFrame(rows)
    
    # Print to console
    print("\n--- Results DataFrame ---")
    print(results_df[["Model", "Recall_mean", "Precision_mean", "AUROC_mean"]])
    
    # Generate LaTeX
    print("\n--- LaTeX Table ---")
    print(make_latex_table(results_df))
else:
    print("No results generated (check model errors).")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# PyOD imports
from pyod.models.iforest import IForest
from pyod.models.lof import LOF
from pyod.models.ecod import ECOD
from pyod.models.knn import KNN
from pyod.models.suod import SUOD

# ==========================================================
# METRIC COMPUTATION
# ==========================================================
def compute_metrics(y_true, is_out, scores):
    """
    is_out = 1 for anomaly, 0 for normal
    """
    recall = np.mean(is_out[y_true == 1]) if np.sum(y_true == 1) > 0 else 0.0
    precision = np.mean(y_true[is_out == 1] == 1) if np.sum(is_out == 1) > 0 else 0.0
    fpr = np.mean(is_out[y_true == 0]) if np.sum(y_true == 0) > 0 else 0.0
    inlier_ret = np.mean(is_out[y_true == 0] == 0) if np.sum(y_true == 0) > 0 else 0.0
    auroc = roc_auc_score(y_true, scores) if len(np.unique(y_true)) == 2 else 0.5
    return recall, precision, fpr, inlier_ret, auroc

# ==========================================================
# BENCHMARK FUNCTION (MODIFIED FOR SINGLE DATASET)
# ==========================================================
def run_benchmark_on_data(X, y, dataset_name, models, scaler=StandardScaler(), runs=5):
    """
    Runs models on pre-loaded X and y arrays.
    """
    
    # Scale the data
    X_scaled = scaler.fit_transform(X)
    
    rows = []

    for mdl_name, mdl_fn in models.items():
        print(f"  Running model: {mdl_name}...")

        # Store the metric history across runs
        rec_list, prec_list, fpr_list, inret_list, auc_list = [], [], [], [], []

        for r in range(runs):
            try:
                model = mdl_fn()

                # ---------------------------------------------
                # Special case: VSCOUT / VAE (Keras/Tensorflow models)
                # ---------------------------------------------
                if mdl_name.upper() in ["VSCOUT", "VAE"]:
                    # Assuming fit signature is similar for both
                    model.fit(X_scaled, epochs=110, batch_size=64, verbose=0)

                    out = model.is_outlier(X_scaled)
                    
                    # Handle tuple return (is_outlier, ...) vs single array
                    if isinstance(out, tuple):
                        is_out = np.array(out[0]).astype(int)
                    else:
                        is_out = np.array(out).astype(int)

                    # Scores
                    if hasattr(model, "decision_function"):
                        scores = model.decision_function(X_scaled)
                    else:
                        # Fallback if decision_function not implemented
                        scores = is_out.astype(float)

                # ---------------------------------------------
                # Standard PyOD models
                # ---------------------------------------------
                else:
                    model.fit(X_scaled)

                    if hasattr(model, "predict"):
                        is_out = model.predict(X_scaled).astype(int)
                    else:
                        raise ValueError(f"No usable predict() for {mdl_name}")

                    if hasattr(model, "decision_function"):
                        scores = model.decision_function(X_scaled)
                    else:
                        scores = is_out.astype(float)

                # Compute metrics
                recall, precision, fpr, inlier_ret, auroc = compute_metrics(y, is_out, scores)

                rec_list.append(recall)
                prec_list.append(precision)
                fpr_list.append(fpr)
                inret_list.append(inlier_ret)
                auc_list.append(auroc)
            
            except Exception as e:
                print(f"    Error running {mdl_name} run {r}: {e}")
                continue

        # If all runs failed, skip adding row
        if not rec_list:
            continue

        # Store means + stds
        rows.append(dict(
            Dataset=dataset_name,
            Model=mdl_name,
            Recall_mean=np.mean(rec_list), Recall_std=np.std(rec_list),
            Precision_mean=np.mean(prec_list), Precision_std=np.std(prec_list),
            FPR_mean=np.mean(fpr_list), FPR_std=np.std(fpr_list),
            InlierRetention_mean=np.mean(inret_list), InlierRetention_std=np.std(inret_list),
            AUROC_mean=np.mean(auc_list), AUROC_std=np.std(auc_list),
        ))

    return rows

# ==========================================================
# LATEX TABLE GENERATOR
# ==========================================================
def make_latex_table(results_df):
    def fmt(mean, std):
        return f"{mean:.4f} ({std:.2f})"

    L = []
    L.append("\\begin{table}[H]")
    L.append("\\centering")
    L.append("\\scriptsize")
    L.append("\\caption{Comparison on Single Dataset.}")
    L.append("\\label{tab:single-results}")
    L.append("\\resizebox{0.8\\textwidth}{!}{")
    L.append("\\begin{tabular}{lccccc}")
    L.append("\\toprule")
    L.append("Model & Recall & Precision & FPR & Inlier Retention & AUROC \\\\")
    L.append("\\midrule")

    for _, row in results_df.iterrows():
        L.append(
            f"{row['Model']} & "
            f"{fmt(row['Recall_mean'], row['Recall_std'])} & "
            f"{fmt(row['Precision_mean'], row['Precision_std'])} & "
            f"{fmt(row['FPR_mean'], row['FPR_std'])} & "
            f"{fmt(row['InlierRetention_mean'], row['InlierRetention_std'])} & "
            f"{fmt(row['AUROC_mean'], row['AUROC_std'])} \\\\"
        )

    L.append("\\bottomrule")
    L.append("\\end{tabular}")
    L.append("}")
    L.append("\\end{table}")

    return "\n".join(L)


# ==========================================================
# MAIN EXECUTION
# ==========================================================

models = {
    "VSCOUT": lambda: VSCOUT(alpha=0.05, flag_rule="any"),
    "VAE": lambda: VAE(alpha=0.05),
    "IForest": lambda: IForest(contamination=0.05),
    "LOF": lambda: LOF(contamination=0.05),
    "ECOD": lambda: ECOD(contamination=0.05),
    "KNN": lambda: KNN(contamination=0.05),
    "SUOD": lambda: SUOD(
        base_estimators=[
            IForest(contamination=0.05),
            LOF(contamination=0.05),
            ECOD(contamination=0.05),
            KNN(contamination=0.05),
        ],
        n_jobs=1,
        combination="average",
        contamination=0.05
    )
}


dataset_name = "SECOM"

# 3. RUN BENCHMARK
print(f"Starting benchmark for {dataset_name}...")
rows = run_benchmark_on_data(X, y, dataset_name, models, runs=10)

# 4. DISPLAY RESULTS
if rows:
    results_df = pd.DataFrame(rows)
    
    # Print to console
    print("\n--- Results DataFrame ---")
    print(results_df[["Model", "Recall_mean", "Precision_mean", "AUROC_mean"]])
    
    # Generate LaTeX
    print("\n--- LaTeX Table ---")
    print(make_latex_table(results_df))
else:
    print("No results generated (check model errors).")